# Fase 2: Compresión SVD de BERT fine-tuned

Aplicamos descomposición SVD (Singular Value Decomposition) a las capas lineales del
modelo BERT fine-tuned en GoEmotions para analizar el trade-off calidad/compresión.

**Ejes de análisis:**
1. Espectro de valores singulares — ¿dónde está la redundancia?
2. Compresión uniforme — baseline con rango fijo en todas las capas
3. Por tipo de componente — Q, K, V, Attn Out, FFN Up, FFN Down
4. Atención vs FFN con rangos independientes — grid search de la receta óptima
5. Compresión adaptativa guiada por energía — rango por capa automático
6. Por profundidad — capas tempranas vs tardías
7. Fine-tuning post-SVD — ¿cuánto se recupera?

In [ ]:
# Instalación de dependencias (para Google Colab)
# !pip install transformers datasets torch scikit-learn accelerate seaborn

In [ ]:
import sys
import os

# En Colab, montar Drive y ajustar path al proyecto
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_ROOT = '/content/drive/MyDrive/transformer-structural-compression-v2'

# En local
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

from src.data import load_goemotions
from src.data.dataset import NUM_LABELS, EMOTION_NAMES
from src.utils import compute_metrics
from src.compression import (
    apply_svd_compression,
    compute_adaptive_ranks,
    compute_singular_value_energy,
    count_parameters,
    filter_layer_names,
    get_compression_ratio,
    get_target_layer_names,
)

sns.set_style("whitegrid")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 1. Cargar modelo fine-tuned y datos de test

In [ ]:
model_path = os.path.join(PROJECT_ROOT, "results", "bert-goemotions-final")

baseline_model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
)
baseline_model.eval()

params = count_parameters(baseline_model)
print(f"Parámetros totales: {params['total']:,}")
print(f"Parámetros entrenables: {params['trainable']:,}")

In [ ]:
_, _, test_ds, emotion_names, data_collator = load_goemotions()
print(f"Test set: {len(test_ds)} ejemplos")

In [ ]:
# Helper de evaluación reutilizable
eval_args = TrainingArguments(
    output_dir=os.path.join(PROJECT_ROOT, "results", "tmp_eval"),
    per_device_eval_batch_size=64,
    fp16=(device == "cuda"),
    report_to="none",
)

def evaluate_model(model):
    """Evalúa un modelo en el test set y devuelve el dict de métricas."""
    trainer = Trainer(
        model=model,
        args=eval_args,
        compute_metrics=compute_metrics,
        data_collator=data_collator,
    )
    return trainer.evaluate(test_ds)

In [ ]:
# Verificar que el baseline reproduce las métricas de Fase 1
baseline_metrics = evaluate_model(baseline_model)
baseline_f1_macro = baseline_metrics["eval_f1_macro"]
baseline_f1_micro = baseline_metrics["eval_f1_micro"]
print(f"Baseline F1 macro: {baseline_f1_macro:.4f}")
print(f"Baseline F1 micro: {baseline_f1_micro:.4f}")

In [ ]:
# Nombres de todas las capas target y sus grupos
all_target_names = get_target_layer_names(baseline_model)
print(f"Total capas target: {len(all_target_names)}")
print(f"\nEjemplo (bloque 0):")
for n in all_target_names[:6]:
    print(f"  {n}")

## 2. Análisis del espectro de valores singulares

In [ ]:
energy_info = compute_singular_value_energy(baseline_model)

# Tabla resumen de rangos necesarios para distintos umbrales de energía
rows = []
for name, info in energy_info.items():
    short_name = name.replace("bert.encoder.", "")
    rows.append({
        "layer": short_name,
        "rank_90%": info["rank_for_90"],
        "rank_95%": info["rank_for_95"],
        "rank_99%": info["rank_for_99"],
    })

energy_df = pd.DataFrame(rows)
print(f"Total capas target: {len(energy_df)}")
print(f"\nRango medio para 90% energía: {energy_df['rank_90%'].mean():.0f}")
print(f"Rango medio para 95% energía: {energy_df['rank_95%'].mean():.0f}")
print(f"Rango medio para 99% energía: {energy_df['rank_99%'].mean():.0f}")
energy_df

In [ ]:
# Heatmap: rango para 90% de energía por bloque y componente
component_types = ["attention.self.query", "attention.self.key", "attention.self.value",
                   "attention.output.dense", "intermediate.dense", "output.dense"]
component_short = ["Q", "K", "V", "Attn Out", "FFN Up", "FFN Down"]
num_layers = 12

heatmap_data = np.zeros((num_layers, len(component_types)))
for name, info in energy_info.items():
    for j, comp in enumerate(component_types):
        if comp in name:
            layer_idx = int(name.split(".")[3])
            heatmap_data[layer_idx, j] = info["rank_for_90"]
            break

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(heatmap_data, annot=True, fmt=".0f", cmap="YlOrRd",
            xticklabels=component_short,
            yticklabels=[f"Layer {i}" for i in range(num_layers)],
            ax=ax)
ax.set_title("Rango necesario para 90% de energía por capa/componente")
ax.set_xlabel("Componente")
ax.set_ylabel("Capa del encoder")
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "results", "svd_energy_heatmap.png"), dpi=150)
plt.show()

In [ ]:
# Curvas de decaimiento de valores singulares por tipo de componente
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for j, (comp, short) in enumerate(zip(component_types, component_short)):
    ax = axes[j]
    for name, info in energy_info.items():
        if comp in name:
            layer_idx = int(name.split(".")[3])
            sv = info["singular_values"].numpy()
            ax.plot(sv / sv[0], alpha=0.6, label=f"L{layer_idx}")
    ax.set_title(short)
    ax.set_xlabel("Índice")
    ax.set_ylabel("Valor singular (normalizado)")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=6, ncol=2)

fig.suptitle("Decaimiento de valores singulares por tipo de componente", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "results", "svd_decay_curves.png"), dpi=150)
plt.show()

## 3. Curva analítica de parámetros vs rango

In [ ]:
# Reducción teórica de parámetros sin instanciar modelos
layer_dims = []
original_params_in_targets = 0
for name in all_target_names:
    module = baseline_model
    for p in name.split("."):
        module = getattr(module, p)
    m, n = module.weight.shape  # (out_features, in_features)
    bias_params = m if module.bias is not None else 0
    layer_dims.append((m, n, bias_params))
    original_params_in_targets += m * n + bias_params

non_target_params = count_parameters(baseline_model)["total"] - original_params_in_targets

ranks = list(range(1, 769))
total_params_by_rank = []
for rank in ranks:
    compressed_in_targets = sum(
        n * min(rank, min(m, n)) + m * min(rank, min(m, n)) + bias
        for m, n, bias in layer_dims
    )
    total_params_by_rank.append(non_target_params + compressed_in_targets)

original_total = count_parameters(baseline_model)["total"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(ranks, [p / 1e6 for p in total_params_by_rank])
ax1.axhline(y=original_total / 1e6, color="red", linestyle="--", label="Original")
ax1.set_xlabel("Rango SVD")
ax1.set_ylabel("Parámetros totales (M)")
ax1.set_title("Parámetros totales vs rango SVD")
ax1.legend()

ax2.plot(ranks, [p / original_total * 100 for p in total_params_by_rank])
ax2.axhline(y=100, color="red", linestyle="--", label="100% (original)")
for r in [64, 128, 256, 384, 512]:
    pct = total_params_by_rank[r - 1] / original_total * 100
    ax2.plot(r, pct, "o", markersize=6)
    ax2.annotate(f"r={r}\n{pct:.1f}%", (r, pct), textcoords="offset points",
                 xytext=(5, 10), fontsize=8)
ax2.set_xlabel("Rango SVD")
ax2.set_ylabel("% de parámetros originales")
ax2.set_title("Retención de parámetros vs rango SVD")
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "results", "svd_params_vs_rank.png"), dpi=150)
plt.show()

## 4. Compresión uniforme — todas las capas al mismo rango

In [ ]:
svd_ranks = [64, 128, 256, 384, 512]
uniform_results = []

for rank in svd_ranks:
    print(f"\nSVD rank = {rank}", end=" ... ")
    compressed = apply_svd_compression(baseline_model, rank=rank)
    compressed.to(device)

    metrics = evaluate_model(compressed)
    comp_params = count_parameters(compressed)
    ratio = get_compression_ratio(baseline_model, compressed)

    row = {
        "rank": rank,
        "params": comp_params["total"],
        "compression_ratio": ratio,
        "f1_macro": metrics["eval_f1_macro"],
        "f1_micro": metrics["eval_f1_micro"],
    }
    for i, name in enumerate(emotion_names):
        row[f"f1_{name}"] = metrics[f"eval_f1_label_{i}"]
    uniform_results.append(row)

    print(f"F1m={metrics['eval_f1_macro']:.4f} | ratio={ratio:.2f}x")
    del compressed
    if device == "cuda":
        torch.cuda.empty_cache()

uniform_df = pd.DataFrame(uniform_results)
uniform_df["f1_macro_retention"] = uniform_df["f1_macro"] / baseline_f1_macro * 100
uniform_df["f1_micro_retention"] = uniform_df["f1_micro"] / baseline_f1_micro * 100
uniform_df[["rank", "params", "compression_ratio", "f1_macro", "f1_micro",
            "f1_macro_retention", "f1_micro_retention"]]

In [ ]:
# F1 macro/micro vs rango SVD
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(uniform_df["rank"], uniform_df["f1_macro"], "o-", label="F1 macro", linewidth=2)
ax.plot(uniform_df["rank"], uniform_df["f1_micro"], "s-", label="F1 micro", linewidth=2)
ax.axhline(y=baseline_f1_macro, color="C0", linestyle="--", alpha=0.4, label=f"Baseline macro ({baseline_f1_macro:.4f})")
ax.axhline(y=baseline_f1_micro, color="C1", linestyle="--", alpha=0.4, label=f"Baseline micro ({baseline_f1_micro:.4f})")
ax.set_xlabel("Rango SVD")
ax.set_ylabel("F1 Score")
ax.set_title("Compresión uniforme — F1 vs Rango SVD")
ax.legend()
ax.set_xticks(svd_ranks)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "results", "svd_uniform_f1_vs_rank.png"), dpi=150)
plt.show()

In [ ]:
# Heatmap de F1 por emoción a distintos rangos
emotion_cols = [f"f1_{name}" for name in emotion_names]
heatmap_df = uniform_df.set_index("rank")[emotion_cols].copy()
heatmap_df.columns = emotion_names

# Agregar baseline
baseline_row = {name: baseline_metrics[f"eval_f1_label_{i}"] for i, name in enumerate(emotion_names)}
heatmap_df = pd.concat([pd.DataFrame([baseline_row], index=["baseline"]), heatmap_df])

fig, ax = plt.subplots(figsize=(18, 5))
sns.heatmap(heatmap_df, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1,
            ax=ax, linewidths=0.5)
ax.set_title("F1 por emoción — compresión uniforme")
ax.set_xlabel("Emoción")
ax.set_ylabel("Rango SVD")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "results", "svd_uniform_f1_per_emotion.png"), dpi=150)
plt.show()

## 5. Compresión por tipo de componente

Comprimimos **solo** un tipo de componente a la vez para ver cuál tolera más la compresión.
Componentes: Q, K, V, Attn Output, FFN Intermediate (Up), FFN Output (Down).

In [ ]:
components = [
    ("query", "Q"),
    ("key", "K"),
    ("value", "V"),
    ("attention_output", "Attn Out"),
    ("intermediate", "FFN Up"),
    ("ffn_output", "FFN Down"),
]
component_test_ranks = [64, 128, 256]

component_results = []

for comp_key, comp_label in components:
    layer_names = filter_layer_names(all_target_names, component=comp_key)
    print(f"\n{comp_label} ({len(layer_names)} capas)")

    for rank in component_test_ranks:
        compressed = apply_svd_compression(baseline_model, rank=rank, layer_names=layer_names)
        compressed.to(device)

        metrics = evaluate_model(compressed)
        comp_params = count_parameters(compressed)
        ratio = get_compression_ratio(baseline_model, compressed)

        component_results.append({
            "component": comp_label,
            "rank": rank,
            "params": comp_params["total"],
            "compression_ratio": ratio,
            "f1_macro": metrics["eval_f1_macro"],
            "f1_micro": metrics["eval_f1_micro"],
            "f1_macro_retention": metrics["eval_f1_macro"] / baseline_f1_macro * 100,
        })
        print(f"  rank={rank:>3d} | F1m={metrics['eval_f1_macro']:.4f} ({metrics['eval_f1_macro']/baseline_f1_macro*100:.1f}%) | ratio={ratio:.2f}x")

        del compressed
        if device == "cuda":
            torch.cuda.empty_cache()

component_df = pd.DataFrame(component_results)
component_df

In [ ]:
# Visualización: retención de F1 por componente y rango
fig, ax = plt.subplots(figsize=(12, 6))

for comp_key, comp_label in components:
    subset = component_df[component_df["component"] == comp_label]
    ax.plot(subset["rank"], subset["f1_macro_retention"], "o-", label=comp_label, linewidth=2)

ax.axhline(y=100, color="gray", linestyle="--", alpha=0.5, label="Baseline (100%)")
ax.set_xlabel("Rango SVD")
ax.set_ylabel("Retención F1 macro (%)")
ax.set_title("Sensibilidad a compresión por tipo de componente")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
ax.set_xticks(component_test_ranks)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "results", "svd_component_sensitivity.png"), dpi=150)
plt.show()

In [ ]:
# También: compresión de grupos — solo atención vs solo FFN
group_results = []

for group, group_label in [("attention", "Solo Atención"), ("ffn", "Solo FFN")]:
    layer_names = filter_layer_names(all_target_names, component=group)
    print(f"\n{group_label} ({len(layer_names)} capas)")

    for rank in component_test_ranks:
        compressed = apply_svd_compression(baseline_model, rank=rank, layer_names=layer_names)
        compressed.to(device)

        metrics = evaluate_model(compressed)
        ratio = get_compression_ratio(baseline_model, compressed)

        group_results.append({
            "group": group_label,
            "rank": rank,
            "f1_macro": metrics["eval_f1_macro"],
            "f1_macro_retention": metrics["eval_f1_macro"] / baseline_f1_macro * 100,
            "compression_ratio": ratio,
        })
        print(f"  rank={rank:>3d} | F1m={metrics['eval_f1_macro']:.4f} ({metrics['eval_f1_macro']/baseline_f1_macro*100:.1f}%) | ratio={ratio:.2f}x")

        del compressed
        if device == "cuda":
            torch.cuda.empty_cache()

group_df = pd.DataFrame(group_results)
group_df

## 6. Grid search: rangos independientes para Atención y FFN

La "receta óptima": ¿qué combinación de rango para atención y rango para FFN maximiza compresión con mínima pérdida?

In [ ]:
attn_ranks = [64, 128, 256, 384]
ffn_ranks = [128, 256, 384, 512]

attn_names = filter_layer_names(all_target_names, component="attention")
ffn_names = filter_layer_names(all_target_names, component="ffn")

grid_results = []

for attn_rank in attn_ranks:
    for ffn_rank in ffn_ranks:
        # Construir dict de rangos por capa
        rank_map = {}
        for n in attn_names:
            rank_map[n] = attn_rank
        for n in ffn_names:
            rank_map[n] = ffn_rank

        compressed = apply_svd_compression(baseline_model, rank=rank_map)
        compressed.to(device)

        metrics = evaluate_model(compressed)
        ratio = get_compression_ratio(baseline_model, compressed)

        grid_results.append({
            "attn_rank": attn_rank,
            "ffn_rank": ffn_rank,
            "f1_macro": metrics["eval_f1_macro"],
            "f1_macro_retention": metrics["eval_f1_macro"] / baseline_f1_macro * 100,
            "compression_ratio": ratio,
            "params": count_parameters(compressed)["total"],
        })
        print(f"attn={attn_rank:>3d} ffn={ffn_rank:>3d} | F1m={metrics['eval_f1_macro']:.4f} ({metrics['eval_f1_macro']/baseline_f1_macro*100:.1f}%) | ratio={ratio:.2f}x")

        del compressed
        if device == "cuda":
            torch.cuda.empty_cache()

grid_df = pd.DataFrame(grid_results)
grid_df

In [ ]:
# Heatmap de F1 macro retención para el grid atención × FFN
pivot_f1 = grid_df.pivot(index="attn_rank", columns="ffn_rank", values="f1_macro_retention")
pivot_ratio = grid_df.pivot(index="attn_rank", columns="ffn_rank", values="compression_ratio")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(pivot_f1, annot=True, fmt=".1f", cmap="RdYlGn", ax=ax1,
            vmin=pivot_f1.values.min() - 5, vmax=100)
ax1.set_title("Retención F1 macro (%)")
ax1.set_xlabel("Rango FFN")
ax1.set_ylabel("Rango Atención")

sns.heatmap(pivot_ratio, annot=True, fmt=".2f", cmap="YlOrRd_r", ax=ax2)
ax2.set_title("Ratio de compresión (x)")
ax2.set_xlabel("Rango FFN")
ax2.set_ylabel("Rango Atención")

fig.suptitle("Grid search: Rango Atención × Rango FFN", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "results", "svd_grid_attn_ffn.png"), dpi=150)
plt.show()

In [ ]:
# Mejor configuración: máxima compresión con >95% retención de F1
viable = grid_df[grid_df["f1_macro_retention"] >= 95].sort_values("compression_ratio", ascending=False)
if len(viable) > 0:
    best = viable.iloc[0]
    print(f"Mejor configuración con >=95% retención:")
    print(f"  Atención rank: {int(best['attn_rank'])}")
    print(f"  FFN rank:      {int(best['ffn_rank'])}")
    print(f"  F1 macro:      {best['f1_macro']:.4f} ({best['f1_macro_retention']:.1f}%)")
    print(f"  Compresión:    {best['compression_ratio']:.2f}x")
else:
    print("Ninguna configuración del grid alcanza 95% retención.")
    best = grid_df.sort_values("f1_macro_retention", ascending=False).iloc[0]
    print(f"Mejor disponible: attn={int(best['attn_rank'])}, ffn={int(best['ffn_rank'])} → {best['f1_macro_retention']:.1f}%")

## 7. Compresión adaptativa guiada por energía

Cada capa recibe el rango mínimo para retener un umbral de energía.
Estrategia "inteligente" vs la uniforme.

In [ ]:
energy_thresholds = [0.80, 0.85, 0.90, 0.95, 0.99]
adaptive_results = []

for threshold in energy_thresholds:
    adaptive_ranks = compute_adaptive_ranks(energy_info, energy_threshold=threshold)
    avg_rank = np.mean(list(adaptive_ranks.values()))
    min_rank = min(adaptive_ranks.values())
    max_rank = max(adaptive_ranks.values())

    compressed = apply_svd_compression(baseline_model, rank=adaptive_ranks)
    compressed.to(device)

    metrics = evaluate_model(compressed)
    ratio = get_compression_ratio(baseline_model, compressed)

    adaptive_results.append({
        "threshold": f"{threshold:.0%}",
        "avg_rank": avg_rank,
        "min_rank": min_rank,
        "max_rank": max_rank,
        "params": count_parameters(compressed)["total"],
        "compression_ratio": ratio,
        "f1_macro": metrics["eval_f1_macro"],
        "f1_micro": metrics["eval_f1_micro"],
        "f1_macro_retention": metrics["eval_f1_macro"] / baseline_f1_macro * 100,
    })
    print(f"Energía {threshold:.0%} | rango medio={avg_rank:.0f} [{min_rank}-{max_rank}] | "
          f"F1m={metrics['eval_f1_macro']:.4f} ({metrics['eval_f1_macro']/baseline_f1_macro*100:.1f}%) | ratio={ratio:.2f}x")

    del compressed
    if device == "cuda":
        torch.cuda.empty_cache()

adaptive_df = pd.DataFrame(adaptive_results)
adaptive_df

In [ ]:
# Comparar adaptativo vs uniforme
fig, ax = plt.subplots(figsize=(10, 6))

# Uniforme
ax.plot(uniform_df["compression_ratio"], uniform_df["f1_macro_retention"],
        "o-", label="Uniforme", linewidth=2, markersize=8)
for _, row in uniform_df.iterrows():
    ax.annotate(f"r={int(row['rank'])}", (row["compression_ratio"], row["f1_macro_retention"]),
                textcoords="offset points", xytext=(5, 8), fontsize=8)

# Adaptativo
ax.plot(adaptive_df["compression_ratio"], adaptive_df["f1_macro_retention"],
        "s-", label="Adaptativo (energía)", linewidth=2, markersize=8)
for _, row in adaptive_df.iterrows():
    ax.annotate(row["threshold"], (row["compression_ratio"], row["f1_macro_retention"]),
                textcoords="offset points", xytext=(5, -12), fontsize=8)

ax.axhline(y=100, color="gray", linestyle="--", alpha=0.5)
ax.axhline(y=95, color="red", linestyle=":", alpha=0.5, label="95% retención")
ax.set_xlabel("Ratio de compresión (x)")
ax.set_ylabel("Retención F1 macro (%)")
ax.set_title("Uniforme vs Adaptativo: frontera de Pareto")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "results", "svd_uniform_vs_adaptive.png"), dpi=150)
plt.show()

## 8. Análisis por profundidad — capas tempranas vs tardías

In [ ]:
depth_results = []
test_rank = 128

# Comprimir solo capas tempranas (0-5) vs tardías (6-11)
depth_configs = [
    ("Tempranas (0-5)", list(range(0, 6))),
    ("Tardías (6-11)", list(range(6, 12))),
]

# También capa por capa
for i in range(12):
    depth_configs.append((f"Solo capa {i}", [i]))

for label, layer_indices in depth_configs:
    layer_names = filter_layer_names(all_target_names, layers=layer_indices)
    compressed = apply_svd_compression(baseline_model, rank=test_rank, layer_names=layer_names)
    compressed.to(device)

    metrics = evaluate_model(compressed)
    ratio = get_compression_ratio(baseline_model, compressed)

    depth_results.append({
        "config": label,
        "layers": str(layer_indices),
        "n_layers_compressed": len(layer_names),
        "f1_macro": metrics["eval_f1_macro"],
        "f1_macro_retention": metrics["eval_f1_macro"] / baseline_f1_macro * 100,
        "compression_ratio": ratio,
    })
    print(f"{label:>20s} | F1m={metrics['eval_f1_macro']:.4f} ({metrics['eval_f1_macro']/baseline_f1_macro*100:.1f}%) | ratio={ratio:.2f}x")

    del compressed
    if device == "cuda":
        torch.cuda.empty_cache()

depth_df = pd.DataFrame(depth_results)
depth_df

In [ ]:
# Impacto de comprimir cada capa individual (rank=128)
per_layer = depth_df[depth_df["config"].str.startswith("Solo capa")].copy()
per_layer["layer_idx"] = per_layer["config"].str.extract(r"(\d+)").astype(int)
per_layer = per_layer.sort_values("layer_idx")

fig, ax = plt.subplots(figsize=(12, 5))
colors = ["green" if r > 99 else "orange" if r > 95 else "red" for r in per_layer["f1_macro_retention"]]
ax.bar(per_layer["layer_idx"], 100 - per_layer["f1_macro_retention"], color=colors)
ax.set_xlabel("Capa del encoder")
ax.set_ylabel("Caída de F1 macro (%)")
ax.set_title(f"Impacto de comprimir cada capa individualmente (rank={test_rank})")
ax.set_xticks(range(12))
ax.set_xticklabels([f"L{i}" for i in range(12)])
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "results", "svd_per_layer_impact.png"), dpi=150)
plt.show()

## 9. Fine-tuning post-SVD

In [ ]:
train_ds_ft, val_ds_ft, _, _, _ = load_goemotions()

finetune_rank = 128
ft_model = apply_svd_compression(baseline_model, rank=finetune_rank)
ft_model.to(device)

pre_ft_metrics = evaluate_model(ft_model)
print(f"Pre fine-tuning  — F1 macro: {pre_ft_metrics['eval_f1_macro']:.4f}")

In [ ]:
ft_args = TrainingArguments(
    output_dir=os.path.join(PROJECT_ROOT, "results", f"svd-ft-rank{finetune_rank}"),
    num_train_epochs=2,
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    fp16=(device == "cuda"),
    report_to="none",
    seed=42,
)

ft_trainer = Trainer(
    model=ft_model,
    args=ft_args,
    train_dataset=train_ds_ft,
    eval_dataset=val_ds_ft,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

ft_trainer.train()

In [ ]:
post_ft_metrics = evaluate_model(ft_model)

print(f"Resultados fine-tuning post-SVD (rank={finetune_rank}):")
print(f"  Baseline:     F1 macro = {baseline_f1_macro:.4f}")
print(f"  Pre-FT:       F1 macro = {pre_ft_metrics['eval_f1_macro']:.4f}")
print(f"  Post-FT:      F1 macro = {post_ft_metrics['eval_f1_macro']:.4f}")

gap = baseline_f1_macro - pre_ft_metrics['eval_f1_macro']
if gap > 0:
    recovery = (post_ft_metrics['eval_f1_macro'] - pre_ft_metrics['eval_f1_macro']) / gap * 100
    print(f"  Recuperación: {recovery:.1f}% del gap")

## 10. Guardar resultados

In [ ]:
# CSV con todos los experimentos
uniform_df["experiment"] = "uniform"
component_df["experiment"] = "component"
grid_df["experiment"] = "grid_attn_ffn"
adaptive_df["experiment"] = "adaptive"
depth_df["experiment"] = "depth"

uniform_df.to_csv(os.path.join(PROJECT_ROOT, "results", "svd_uniform_results.csv"), index=False)
component_df.to_csv(os.path.join(PROJECT_ROOT, "results", "svd_component_results.csv"), index=False)
grid_df.to_csv(os.path.join(PROJECT_ROOT, "results", "svd_grid_results.csv"), index=False)
adaptive_df.to_csv(os.path.join(PROJECT_ROOT, "results", "svd_adaptive_results.csv"), index=False)
depth_df.to_csv(os.path.join(PROJECT_ROOT, "results", "svd_depth_results.csv"), index=False)
print("CSVs guardados en results/")

In [ ]:
# JSON consolidado
all_metrics = {
    "baseline": {
        "f1_macro": baseline_f1_macro,
        "f1_micro": baseline_f1_micro,
        "params": count_parameters(baseline_model)["total"],
    },
    "uniform": uniform_df[["rank", "params", "compression_ratio", "f1_macro", "f1_micro", "f1_macro_retention"]].to_dict(orient="records"),
    "component": component_df.to_dict(orient="records"),
    "grid_attn_ffn": grid_df.to_dict(orient="records"),
    "adaptive": adaptive_df.to_dict(orient="records"),
    "depth": depth_df.to_dict(orient="records"),
    "post_svd_finetuning": {
        "rank": finetune_rank,
        "pre_ft_f1_macro": pre_ft_metrics["eval_f1_macro"],
        "post_ft_f1_macro": post_ft_metrics["eval_f1_macro"],
    } if "post_ft_metrics" in dir() else None,
}

with open(os.path.join(PROJECT_ROOT, "results", "svd_metrics.json"), "w") as f:
    json.dump(all_metrics, f, indent=2, default=float)
print("Métricas JSON guardadas")

In [ ]:
# Guardar mejor modelo comprimido
best_compressed = apply_svd_compression(baseline_model, rank=finetune_rank)
save_path = os.path.join(PROJECT_ROOT, "results", f"bert-svd-rank{finetune_rank}")
os.makedirs(save_path, exist_ok=True)
torch.save({
    "model_state_dict": best_compressed.state_dict(),
    "rank": finetune_rank,
    "config": best_compressed.config.to_dict(),
}, os.path.join(save_path, "model.pt"))
print(f"Modelo comprimido guardado en: {save_path}")

# Guardar modelo post fine-tuning
if "ft_model" in dir():
    ft_save_path = os.path.join(PROJECT_ROOT, "results", f"bert-svd-rank{finetune_rank}-ft")
    os.makedirs(ft_save_path, exist_ok=True)
    torch.save({
        "model_state_dict": ft_model.state_dict(),
        "rank": finetune_rank,
        "config": ft_model.config.to_dict(),
    }, os.path.join(ft_save_path, "model.pt"))
    print(f"Modelo post-FT guardado en: {ft_save_path}")

In [ ]:
# Resumen final
print("=" * 70)
print("RESUMEN — Fase 2: Compresión SVD")
print("=" * 70)
print(f"\nBaseline: F1 macro = {baseline_f1_macro:.4f}, params = {count_parameters(baseline_model)['total']:,}")

print(f"\n--- Compresión uniforme ---")
for _, row in uniform_df.iterrows():
    print(f"  rank={int(row['rank']):>3d} | {row['compression_ratio']:.2f}x | F1m={row['f1_macro']:.4f} ({row['f1_macro_retention']:.1f}%)")

print(f"\n--- Mejor config grid Atención×FFN ---")
best_grid = grid_df.sort_values("f1_macro_retention", ascending=False).iloc[0]
print(f"  attn={int(best_grid['attn_rank'])}, ffn={int(best_grid['ffn_rank'])} | {best_grid['compression_ratio']:.2f}x | F1m={best_grid['f1_macro']:.4f} ({best_grid['f1_macro_retention']:.1f}%)")

print(f"\n--- Compresión adaptativa ---")
for _, row in adaptive_df.iterrows():
    print(f"  energía={row['threshold']} | rango medio={row['avg_rank']:.0f} | {row['compression_ratio']:.2f}x | F1m={row['f1_macro']:.4f} ({row['f1_macro_retention']:.1f}%)")

if "post_ft_metrics" in dir():
    print(f"\n--- Fine-tuning post-SVD (rank={finetune_rank}) ---")
    print(f"  Pre-FT:  {pre_ft_metrics['eval_f1_macro']:.4f}")
    print(f"  Post-FT: {post_ft_metrics['eval_f1_macro']:.4f}")